# Подготовка данных

Посмотрим на первые пять строк CSV. 

In [ ]:
from pathlib import Path

RENT_PATH = "data/raw/rent_v19.csv"
if Path.cwd().name == "notebooks":
    RENT_PATH = "../" + RENT_PATH

with open(RENT_PATH, encoding="utf-8-sig") as f:
    for _ in range(5):
        print(repr(f.readline()))


## 1. Формат выгрузки

- Кодировка `encoding="utf-8-sig"`
- Столбцы разделены табуляцией (`\t`).  Строки заканчиваются (`\n`).
- Дробные значения записаны через точку.
- `listing_id` читаем как строку, чтобы сохранить начальные нули. Даты записаны в формате `ДД.ММ.ГГГГ`.

In [ ]:
import pandas as pd

df = pd.read_csv(
    RENT_PATH,
    sep="\t",
    encoding="utf-8-sig",
    decimal=".",
    dtype={"listing_id": "string"},
)
df.head()


Видим сразу, что есть этажи квартиры выше, чем максимальный в доме, также в столбике расстояния до метро встречается число 99999 вероятно это обозначение высшей границы.

## 2. Разбиение данных
Удаляем точные дубли и строки без `rent_rub` и разбиваем данные. train/test = 80/20, с валидацией те же проценты.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET = "rent_rub"

raw_df = pd.read_csv(
    RENT_PATH, sep="\t", encoding="utf-8-sig", dtype=str, keep_default_na=False
)
exact_duplicate = raw_df.duplicated()
missing_target = df[TARGET].isna()
eligible = df.loc[~exact_duplicate & ~missing_target]

train_val_idx, test_idx = train_test_split(
    eligible.index, test_size=0.2, random_state=RANDOM_STATE
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.2, random_state=RANDOM_STATE
)
train = df.loc[train_idx].copy()
validation = df.loc[val_idx].copy()

test_path = Path(RENT_PATH).parent.parent / "interim" / "rent_v19_test.csv"
test_path.parent.mkdir(parents=True, exist_ok=True)
raw_df.loc[test_idx].to_csv(
    test_path, sep="\t", encoding="utf-8-sig", lineterminator="\n", index=False
)

display(pd.Series({
    "Исходные строки": len(df),
    "Удалено точных дублей": exact_duplicate.sum(),
    "Удалено строк без цели после дедупликации": (~exact_duplicate & missing_target).sum(),
    "Train": len(train),
    "Validation": len(validation),
    "Test": len(test_idx),
}, name="Строк"))

## 3. Аудит качества на train

In [ ]:
audit = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": train.isna().mean().mul(100),
    "unique": train.nunique(),
    "example": train.iloc[0],
})
display(audit)


In [ ]:
dates = pd.to_datetime(train["publication_date"], format="%d.%m.%Y", errors="coerce")
checks = {
    "Заполненная дата не читается": train["publication_date"].notna() & dates.isna(),
    "Год постройки позже года публикации обьявления": train["year_built"] > dates.dt.year,
    "Дата вне заданного периода": dates.notna() & ~dates.between("2026-09-01", "2026-10-31"),
    "Рейтинг вне диапазона от 1 до 5": train["owner_rating"].notna() & ~train["owner_rating"].between(1, 5),
    "Этаж выше этажности": train["floor"] > train["floors_total"],
    "Жилая площадь плюс площадь кухни больше общей": (train["living_area"] + train["kitchen_area"] - train["total_area"]).round(8) > 0,
    "Жилая площадь больше общей": train["living_area"] > train["total_area"],
    "Площадь кухни больше общей": train["kitchen_area"] > train["total_area"],
}
issue_counts = pd.Series({name: mask.sum() for name, mask in checks.items()})
display(pd.DataFrame({"rows": issue_counts, "pct_of_train": issue_counts / len(train) * 100}))

**Выводы аудита.** На train обнаружено 29 объявлений с этажом выше этажности дома, тк их и 30 объявлений с жилой площадью больше общей. У рейтинга арендодателя 3 146 пропусков из 7 777 (40,5%). Все заполненные даты разобрались и попали в заявленный период.

In [ ]:
category_candidates = [
    "district", "house_type", "elevator", "rooms", "renovation",
    "furniture", "appliances", "balcony", "parking", "pets_allowed",
    "seller_type", "max_tenants",
]
for column in category_candidates:
    counts = train[column].value_counts(dropna=False)
    display(counts.rename_axis(column).to_frame("n_rows"))

Видим, что в типе дома записано одно и то же, но разными вариантами, надо нормализовать, также в записи количества жильцов, года постройки и балконов можно дробную часть не использовать

## 4. Разведочный анализ на train

In [ ]:
iqr_columns = [
    "total_area",
    "living_area",
    "kitchen_area",
    "ceiling_height",
    "metro_distance_m",
    "distance_to_center_km",
    "response_time_hours",
]
quantitative_columns = iqr_columns + [
    "year_built", "floor", "floors_total",
]
display(train[quantitative_columns].describe().T.drop(columns=["count", "std"]))


### Выводы по описательной статистике

- **Расстояние до метро.** Значение `99999` встречается в 2 281 из 7 777 строк (29,3%). Среди остальных значений максимум — 6 000 м. Такой разрыв указывает на вероятный служебный код, который заметно завышает среднее. Перед обработкой нужно уточнить смысл этого значения.
- **Площадь и высота потолков.** Медиана общей площади — 47,2 м², максимум — 1 393 м². Для жилой площади это 26,6 и 792,3 м², для потолков — 2,67 и 6,99 м. Крайние значения стоит проверить по отдельным объявлениям перед выбором правила обработки.
- **Время ответа.** Медиана — 3 часа, 75% заполненных значений не превышают 5,9 часа, максимум — 87,4 часа. Долгий ответ возможен, поэтому такие значения стоит сначала изучить.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
def plot_rent_boxplots(ax, grouped_rent, medians, labels):
    positions = range(1, len(medians) + 1)
    values = [grouped_rent.get_group(group) for group in medians.index]
    ax.boxplot(values, orientation="horizontal", positions=positions)
    ax.set_yticks(positions, labels=labels)
    for position, median in zip(positions, medians):
        ax.annotate(
            f"{median:,.0f}".replace(",", " "), (median, position),
            xytext=(4, 0), textcoords="offset points", va="center",
            fontsize=8, bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8},
        )


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(data=train, x=TARGET, bins=40, ax=axes[0])
axes[0].set_title("Аренда на train")
axes[0].set_xlabel("Аренда, руб./мес.")
district_rent = train.groupby("district")[TARGET]
district_medians = district_rent.median().sort_values()
plot_rent_boxplots(axes[1], district_rent, district_medians, district_medians.index)
axes[1].set(title="Аренда по районам", xlabel='Аренда, руб./мес.')
plt.tight_layout()
plt.show()

Скошено распределение целевой

In [ ]:
binary_columns = ["elevator", "appliances", "pets_allowed", "balcony"]
rent_numeric = train.select_dtypes(include="number").drop(columns=binary_columns, errors="ignore")
rooms_numeric = train["rooms"].str.extract(r"^(\d+)-", expand=False).astype(float)
rooms_numeric = rooms_numeric.mask(train["rooms"].eq("студия"), 0)

matrix_data = rent_numeric.assign(rooms_numeric=rooms_numeric)
correlation = matrix_data.corr(method="spearman")
column_order = correlation[TARGET].abs().sort_values(ascending=False).index
correlation = correlation.loc[column_order, column_order]

def plot_correlation_heatmap(correlation, title, label):
    fig, ax = plt.subplots(figsize=(13, 11), layout="constrained")
    sns.heatmap(
        correlation, annot=True, fmt=".2f", cmap="RdBu_r",
        vmin=-1, vmax=1, square=True, cbar_kws={"label": label}, ax=ax,
    )
    plt.setp(ax.get_xticklabels(), rotation=55, ha="right")
    ax.set_title(title)
    plt.show()


plot_correlation_heatmap(correlation, 'Ранговые корреляции на train', 'Корреляция Спирмена')


In [ ]:
pearson_correlation = matrix_data.corr(method="pearson").loc[column_order, column_order]
plot_correlation_heatmap(pearson_correlation, 'Корреляция Пирсона на train', 'Корреляция Пирсона')


Смотрим на корреляцию спирмана, видим, что рейтиг продавца и время ответа почти не влияет ни на один признак, признаки малоинформативны

### Мультиколлинеарность числовых признаков (VIF)

Для VIF исключаем `rent_rub` и `owner_rating`. Значение `99999` в `metro_distance_m` считаем пропуском только в таблице для VIF. В расчёт входят строки, где заполнены все выбранные признаки.


In [ ]:
from sklearn.linear_model import LinearRegression

vif_data = matrix_data.drop(columns=[TARGET, "owner_rating"])
vif_data["metro_distance_m"] = vif_data["metro_distance_m"].replace(99999, float("nan"))
vif_data = vif_data.dropna()
vif_values = {}
for column in vif_data.columns:
    predictors = vif_data.drop(columns=column)
    feature = vif_data[column]
    model = LinearRegression().fit(predictors, feature)
    r_squared = model.score(predictors, feature)
    vif_values[column] = float("inf") if r_squared >= 1 else 1 / (1 - r_squared)

print(f"Строк в расчёте: {len(vif_data)} из {len(train)}")
vif = pd.Series(vif_values, name="VIF").sort_values(ascending=False)
display(vif.round(2).to_frame())


In [ ]:
area_columns = {
    "total_area": "Общая площадь",
    "living_area": "Жилая площадь",
    "kitchen_area": "Площадь кухни",
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, layout="constrained")
for ax, (column, title) in zip(axes, area_columns.items()):
    sns.scatterplot(data=train, x=column, y=TARGET, alpha=0.2, s=8, ax=ax)
    ax.set(ylabel="", xlabel=f"{title}, м²", title=title)
fig.supylabel('Аренда, руб./мес.')
plt.show()


Видим, что связь аренды и площади жилой зоны и общей площади одинаковы

In [ ]:
discrete_features = [
    column for column in rent_numeric.columns.drop(TARGET)
    if train[column].nunique() <= 10
]
text_features = train.select_dtypes(include=["object", "string", "category"]).columns
category_features = [
    column for column in [*text_features, *binary_columns, *discrete_features]
    if column not in {"listing_id", "publication_date",  "house_type"}
]
for column in category_features:
    labels = train[column].astype("string").fillna("Пропуск")
    grouped_rent = train[TARGET].groupby(labels)
    medians = grouped_rent.median().sort_values()
    ordered_labels = medians.index
    fig, ax = plt.subplots(figsize=(9, max(2.5, len(ordered_labels) * 0.34)), layout="constrained")
    tick_labels = [
        f"{label!r} (n={len(grouped_rent.get_group(label))})" for label in ordered_labels
    ]
    plot_rent_boxplots(ax, grouped_rent, medians, tick_labels)
    ax.set(title=column, xlabel='Аренда, руб./мес.')
    plt.show()


Видим, что на медиану целевой сильно влияет ремонт, колво комнат, колво жильцов. Интересно, что при 0 балконов медиана цены выше. Животные почти не влияют на целевую, как и балконы.

In [ ]:
floor_groups = pd.cut(
    train["floor"], bins=[0, 4, 9, 14, 19, float("inf")],
    labels=["1–4", "5–9", "10–14", "15–19", "20+"],
)
floor_rent_medians = (
    train.groupby([floor_groups, "balcony"], observed=True)[TARGET]
    .median()
    .unstack()
    .rename(columns=lambda count: str(int(count)))
)
ax = floor_rent_medians.plot.bar(figsize=(12, 5), rot=0)
ax.set(xlabel="Диапазон этажей", ylabel="Медианная стоимость аренды, ₽/мес.")
ax.legend(title="Число балконов")
plt.tight_layout()
plt.show()


Число балконов не завит от колва этажей

## 5. Решения перед Pipeline
## Пропуски

In [ ]:
missing_count = train.isna().sum()
missing_count = missing_count[missing_count > 0].sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": (missing_count / len(train) * 100).round(2),
})
display(missing_summary)


1. не заполняем
2. мода по ремонту
3. студии
4. медиана по общей или комнатам
5. медиана по году постройки
6. не указан
7. не указан
8. по этажности
9. не указан
10. не заполняем
11. по комнатам мода
12. не указан

### Закономерности пропусков на train

Посмотрим количество пропусков по типу арендодателя, по районам и в группах с заполненным или пропущенным значением связанного поля.

In [ ]:
house_mapping = {
    'панельный': 'панельный',
    'панель': 'панельный',
    'панельный дом': 'панельный',
    'панельн.': 'панельный',

    'кирпичный': 'кирпичный',
    'кирпич': 'кирпичный',
    'кирпичный дом': 'кирпичный',

    'монолитный': 'монолитный',
    'монолит': 'монолитный',
    'монолитный дом': 'монолитный',

    'блочный': 'блочный',
    'блочн.': 'блочный',
    'блочный дом': 'блочный',

    'деревянный': 'деревянный',
    'деревянный дом': 'деревянный',
    'дерево': 'деревянный',
    'деревянн.': 'деревянный',
    'деревян.': 'деревянный'
}

house_type_normalized = train["house_type"].str.strip().str.lower()
house_type_clean = house_type_normalized.map(house_mapping).fillna(house_type_normalized)


In [ ]:
print("Пропуски в рейтинге по типу арендодателя:")
print(train.groupby("seller_type")["owner_rating"].apply(lambda x: x.isna().sum()), "\n")

print("Пропуски в высоте потолков по типу дома:")
print(train.groupby(house_type_clean, dropna=False)["ceiling_height"].apply(lambda x: x.isna().sum()), "\n")

print("Пропуски в кухонной площади по количеству комнат:")
print(train.groupby("rooms")["kitchen_area"].apply(lambda x: x.isna().sum()), "\n")



Видим, что рейтинг арендодателя не заполнен только у агенств, возможно для агенств в целом нет возможности выставить оценку. Также видим, что площадь кухни не указана только в студии, там просто нет кухни. 

In [ ]:
train["furniture"].value_counts(normalize=True, dropna=False)

In [ ]:
pd.crosstab(
    train["renovation"],
    train["furniture"],
    normalize="index"
).plot.bar(stacked=True, figsize=(10, 4))

plt.show()

In [ ]:
furniture_missing_by_renovation = train.groupby("renovation", dropna=False)["furniture"].agg(
    listings="size",
    missing=lambda values: values.isna().sum(),
)
furniture_missing_by_renovation["missing_pct"] = (
    100 * furniture_missing_by_renovation["missing"]
    / furniture_missing_by_renovation["listings"]
).round(2)

print("Пропуски в меблированности по типу ремонта:")
print(furniture_missing_by_renovation.sort_values("missing_pct", ascending=False))


видно что чем лучше ремонт тем меньше пропусков по мебели, а также, что без ремонта распределение по категориям мебели меняется, это значит, что надо проверить два варианта, первое заполнение модой по типу ремонта и также пометить сам признак не заполнения меблированности он мб важен

In [ ]:
build_decade = (train["year_built"] // 10 * 10).astype("Int64")
ceiling_by_house = train["ceiling_height"].groupby(house_type_clean)
ceiling_by_decade = train["ceiling_height"].groupby(build_decade)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, layout="constrained")
for ax, grouped, title in [
    (axes[0], ceiling_by_house, "По типу дома"),
    (axes[1], ceiling_by_decade, "По десятилетию постройки"),
]:
    summary = grouped.agg(median="median", n="count").dropna(subset=["median"])
    bars = ax.barh(summary.index.astype(str), summary["median"])
    ax.bar_label(
        bars, labels=[f"{row.median:.2f} (n={row.n})" for row in summary.itertuples()],
        padding=3,
    )
    ax.margins(x=0.35)
    ax.set(title=title, xlabel="Медианная высота потолков, м")
plt.show()


Видим, что высота потолков зависит от периода постройки, в старых домах высокие потолки были приняты, надо заполнить медианой по 10-летиям

In [ ]:
year_by_floors = train.groupby("floors_total")["year_built"].median()

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
sns.lineplot(x=year_by_floors.index, y=year_by_floors, marker="o", ax=ax)
ax.set(xlabel="Этажность дома", ylabel="Медианный год постройки")
ax.set_xticks(year_by_floors.index)
plt.show()


In [ ]:
from sklearn.model_selection import KFold

for left, right in [("furniture", "renovation"), ("ceiling_height", "year_built")]:
    print(f"{left} + {right}: {(train[left].isna() & train[right].isna()).sum()}")


## Выбросы

In [ ]:
train.loc[
    train["floor"] > train["floors_total"],
    ["floor", "floors_total", "house_type", "year_built", TARGET]
]

Этажи дома в строках с несогласованностью этажности выглядят адекватно и хорошо описываются годом постройки, поэтому их не трогаем, этажи квартир принимаем за пропуск и заполняем медианой по этажности дома 

In [ ]:
excess = (train["living_area"] + train["kitchen_area"] - train["total_area"]).round(8)
excess_pct = excess / train["total_area"] * 100

excess_pct[excess > 1e-8].describe()

где-то проблема подсчета где-то явно неверный ввод данных

In [ ]:
train.assign(
    excess=excess,
    excess_pct=excess_pct
).nlargest(20, "excess_pct")[
    ["rooms", "total_area", "living_area",
     "kitchen_area", "excess", "excess_pct"]
]

In [ ]:
valid = train[
    (train["living_area"] <= train["total_area"]) &
    (train["living_area"] > 0) &
    (train["total_area"] > 0)
]

valid.groupby("rooms")[["total_area", "living_area"]].median().round(2)

In [ ]:
area_columns = ["living_area", "total_area"]
valid_areas = train.loc[
    excess.le(0) & train[area_columns].gt(0).all(axis=1)
]
areas_by_rooms = valid_areas.groupby("rooms")[area_columns]
area_medians = areas_by_rooms.median()
area_iqr = areas_by_rooms.quantile(0.75) - areas_by_rooms.quantile(0.25)

problem_areas = train.loc[excess.gt(0), ["rooms", *area_columns]]
deviations = pd.DataFrame(index=problem_areas.index)
for column in area_columns:
    median = problem_areas["rooms"].map(area_medians[column])
    spread = problem_areas["rooms"].map(area_iqr[column]).replace(0, float("nan"))
    deviations[column] = (problem_areas[column] - median).abs() / spread

living = deviations["living_area"]
total = deviations["total_area"]
print("Отклонение жилой больше:", living.gt(total).sum())
print("Отклонение общей больше:", total.gt(living).sum())
print("Отклонения равны:", living.eq(total).sum())
print("Нет оценки:", deviations.isna().any(axis=1).sum())


будем менять то, у чего больше отклонение от медианы или чекнем вариант моделирования без жилой зоны, она проблемная. Заполняем как пропуски медианами по комнатам

In [ ]:
for area in ["total_area", "living_area"]:
    grouped = train.groupby("rooms")[area]
    q1 = grouped.quantile(0.25)
    q3 = grouped.quantile(0.75)
    upper = q3 + 1.5 * (q3 - q1)
    flagged = train[area].gt(train["rooms"].map(upper))

    summary = grouped.agg(n="count", median="median", max="max")
    summary["upper_iqr"] = upper
    summary["above_upper"] = flagged.groupby(train["rooms"]).sum()
    print(area)
    display(summary.round(2))



In [ ]:
room_order = ["студия", "1-комн.", "2-комн.", "3-комн.", "4-комн.", "5-комн."]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True, layout="constrained")

for ax, (column, title) in zip(axes, [
    ("total_area", "Общая площадь"),
    ("living_area", "Жилая площадь"),
]):
    area_data = train.loc[train[column].gt(0), ["rooms", column]]
    sns.boxplot(data=area_data, x=column, y="rooms", order=room_order, ax=ax)
    medians = area_data.groupby("rooms")[column].median()
    for position, room in enumerate(room_order):
        median = medians.loc[room]
        ax.annotate(
            f"{median:.1f}", (median, position), xytext=(5, 0),
            textcoords="offset points", va="center", fontsize=8,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8},
        )
    ax.set(xscale="log", xlabel="Площадь, м² (логарифмическая шкала)", ylabel="", title=title)
axes[0].set_ylabel("Комнатность")
plt.show()


проблема только в значениях выше iqr, тех, что меньше мало и выглядят они адекватно на фоне сосеедних распределений по комнатам, например, у нас выбрасов по жилой зоне в 2 комнатных квартирах, но они не выходят за минимум однушки, значит это просто очень маленькая двушка. То же правило можно применить в верхним выбросам, тем, что находятся в пределах усов квартир с большим количеством комнат

In [ ]:
cols = ["total_area", "living_area"]

q1 = train.groupby("rooms")[cols].quantile(0.25)
q3 = train.groupby("rooms")[cols].quantile(0.75)

upper = q3 + 1.5 * (q3 - q1)

room_map = {
    "студия": "2-комн.",
    "1-комн.": "3-комн.",
    "2-комн.": "4-комн.",
    "3-комн.": "5-комн.",
    "4-комн.": "5-комн.",
    "5-комн.": "5-комн."
}

for col in cols:
    limit = train["rooms"].map(room_map).map(upper[col])

    mask = train[col] > limit

    print(f"{col}: {mask.sum()} выбросов")

    display(
        train.loc[mask, [
            "rooms", "total_area", "living_area",
            "kitchen_area", TARGET
        ]]
        .assign(upper_limit=limit[mask])
        .sort_values(col, ascending=False)
        .round(2)
    )

видим, что по жилой площади есть только 4 пропуска и все обработаются при заполнении несогласованности жилой и общей зоны, в первых двух строчках неадекватны оба значения
По общей площади похоже, что данные при вводе были случайно увеличены на порядок, пометим как пропуски и заполним медианой по комнатам

In [ ]:
metro_code = train["metro_distance_m"].eq(99999)
center_ranges = pd.cut(train["distance_to_center_km"], bins=range(0, 19, 2), include_lowest=True)
district_labels = train["district"].fillna("Не указан")

fig, axes = plt.subplots(1, 2, figsize=(15, 5), layout="constrained")
for ax, groups, title in [
    (axes[0], district_labels, "По районам"),
    (axes[1], center_ranges, "По расстоянию до центра, км"),
]:
    summary = metro_code.groupby(groups, observed=True).agg(total="size", code="sum")
    summary["pct"] = 100 * summary["code"] / summary["total"]
    if ax is axes[0]:
        summary = summary.sort_values("pct")
    bars = ax.barh(summary.index.astype(str), summary["pct"])
    labels = [f"{row.pct:.0f}% ({row.code}/{row.total})" for row in summary.itertuples()]
    ax.bar_label(bars, labels=labels, padding=3, fontsize=8)
    ax.set(xlim=(0, 130), xlabel="Доля объявлений с кодом 99999, %", title=title)
    ax.set_xticks([0, 25, 50, 75, 100])
plt.show()


Видим, что 99999 определенно код, вероятно райна в котором нет метро, заменим его на пропуск и добавим флаг наличие метро

In [ ]:
year_groups = (train["year_built"] // 10 * 10).astype("Int64").astype("string")
decade_order = sorted(year_groups.dropna().unique(), key=int)

fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
sns.boxplot(x=train["ceiling_height"], y=year_groups, order=decade_order, ax=ax)
ax.set(xlabel="Высота потолков, м", ylabel="Десятилетие постройки")
plt.show()


In [ ]:
cols = [
    "ceiling_height", "year_built", "house_type",
    "rooms", "total_area", TARGET
]

display(train.nlargest(20, "ceiling_height")[cols])

In [ ]:
sns.histplot(
    train.loc[train["ceiling_height"] > 3.5, "ceiling_height"],
    binwidth=0.1
)

plt.xticks(rotation=45)
plt.show()

видим явный разрыв по высоте, вероятно это именно выбросы, потому что потолков выше 4 метров и ниже 4.5 почти нет в наших данных, будем считать выбросами и заполнять все потолки выше 4 метров как остальные пропуски потолков 

## Будущий Pipeline

### Типы и признаки

- `listing_id` читаем как строку и исключаем из признаков модели. Если используем `publication_date`, разбор формата `%d.%m.%Y` и извлечение признаков вынесем в Pipeline.
- `rooms` пока оставляем категорией. Для `house_type` убираем краевые пробелы, приводим значения к нижнему регистру и объединяем известные варианты написания. `elevator`, `appliances` и `pets_allowed` храним как флаги 0/1, `balcony` и `max_tenants` — как числовые счётчики.
- `99999` в `metro_distance_m` заменяем на пропуск и сохраняем флаг `metro_distance_unknown`. Связь кода с районом подтверждена, а его точный смысл пока неизвестен.
- Перед включением `owner_rating` и `response_time_hours` проверяем, доступны ли эти поля в момент создания объявления. Если доступны, их пользу оцениваем по качеству прогноза. Одной парной корреляции для решения мало. Для вопроса о связи числа балконов с этажом нужны доли 0/1/2 по диапазонам этажей. Текущий график показывает аренду по этим группам.

### Площади, этажи и потолки

- Для деления `total_area` на 10 проверяем каждое исправление. Текущий порог `Q3 + 1,5 × IQR` по более высокой комнатности отмечает 68 строк. После деления в 4 строках сумма жилой и кухни превышает исправленную общую, а в 9 строках полную проверку суммы провести нельзя. Перед реализацией явно выбираем порог. Ранее сравнивали с фактическим верхним усом комнатности на три выше, а текущий код использует границу IQR и преимущественно комнатность на две выше. Автоматическое деление применяем только при правдоподобном результате и согласованности с известными площадями.
- Отдельно проверяем `living_area > total_area` и превышение суммы жилой и кухни над общей. При пропущенной кухне первая проверка выявляет ещё 4 строки. Отклонение от медианы по комнатности в единицах IQR помогает выбрать подозрительное поле. После подстановки медианы повторяем проверки, поскольку простая замена оставила несогласованность в 33 из 148 строк. Для нерешённых случаев задаём запасное правило до реализации. Вариант модели без `living_area` сравним на validation.
- Если `floor > floors_total`, рабочий вариант — считать этаж квартиры пропуском и заполнить медианой по этажности дома. После подстановки проверяем, что этаж укладывается в диапазон дома.
- Для `ceiling_height` общий порог 3,22 м захватывает много типичных значений в старых домах. Выше 5 м есть 106 объявлений, из них 55 в панельных домах. До выбора правила смотрим хвост выше 4 м по типу дома и десятилетию и проверяем, есть ли заметный разрыв.

### Заполнение пропусков

| Признак | Правило для первой версии или вариант для проверки |
|---|---|
| `year_built` | Медиана по этажности, общая медиана для неизвестной группы. |
| `ceiling_height` | Медиана по десятилетию исходного года. Если исходный год пропущен, проверить запасной вариант по очищенному типу дома с общей медианой для неизвестной группы. Год и высота одновременно пропущены в 12 строках. |
| `furniture` | Сравнить «не указано» и моду по ремонту с флагом исходного пропуска. Для неизвестной группы использовать общую моду. Мебель и ремонт одновременно пропущены в 18 строках. |
| `parking`, `renovation`, `district` | Начать с категории «не указано». Для ремонта мода по десятилетию остаётся вариантом для сравнения. |
| `kitchen_area` | Пропуски на train относятся к студиям. Отметить этот случай отдельным флагом и выбрать числовое заполнение для модели. |
| `balcony` | Ноль означает отсутствие балкона. Для пропуска проверить явную отметку или заполнение по train. |

### Обучение и проверка

- Все медианы, моды, пороги, получаемые из данных, и категории кодирования определяем только при `fit` на train. К validation и сырому test применяем готовые правила. Варианты сравниваем на одной validation.
- Из-за правого хвоста `rent_rub` можно сравнить обучение на исходной цене и на логарифме. Для метрик возвращаем прогноз к рублям.
- Перед сдачей зафиксировать версии прямых зависимостей в `requirements.txt` и проверить запуск ноутбуков в чистом окружении.


## 6. Журнал изменений данных

Одна запись описывает одно решение. Первые две записи относятся к очистке перед разбиением. Когда добавим `Pipeline`, перед этим разделом появятся ячейки его настройки и обучения. В журнале укажем, какие преобразования применили, почему и сколько строк они затронули. Для параметров, вычисленных при `fit`, укажем, что они получены на train.

In [ ]:
cleaning_log = pd.DataFrame([
    {
        "field": "Все столбцы",
        "issue": "Точные дубли строк",
        "scope": "Исходная выгрузка",
        "n_rows": int(exact_duplicate.sum()),
        "value_before": "Повторная копия строки",
        "action": "Удалить повторные копии, оставить первое вхождение",
        "value_after": "Повторные копии исключены",
        "rationale": "Одинаковые строки не должны попадать в разные выборки",
    },
    {
        "field": TARGET,
        "issue": "Пропущена стоимость аренды",
        "scope": "Выгрузка после удаления точных дублей",
        "n_rows": int((~exact_duplicate & missing_target).sum()),
        "value_before": "Пропуск в rent_rub",
        "action": "Исключить строку",
        "value_after": "В выборках остались строки с известной стоимостью",
        "rationale": "Для обучения и оценки нужна фактическая стоимость аренды",
    },
])